In [ ]:

import SimpleITK as sitk
import numpy as np
import json
import os
from pathlib import Path

# --- CONFIGURAZIONE PERCORSI ---

BASE_DIR = Path("/path/to/data/CAT08")
OUTPUT_DIR = Path("/path/to/data/CAT08")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Convenzione di orientamento desiderata per le immagini di output
TARGET_ORIENTATION = "RAS"

<h2> Conversione formato immagini opzionale <h2>

In [ ]:

def convert_volume_to_ras_nifti(mhd_path, output_nifti_path):
    """
    Legge un file .mhd, lo riorienta in RAS e lo salva come .nii.gz
    """
    print(f"--- Elaborazione volume: {mhd_path.name} ---")
    
    # 1. Caricamento dell'immagine
    img = sitk.ReadImage(str(mhd_path))
    
    # 2. Cambio Orientamento in RAS
    img_ras = sitk.DICOMOrient(img, 'RAS')
    
    # 3. Controllo dati
    img_ras = sitk.Cast(img_ras, sitk.sitkInt16)
    
    # 4. Scrittura del file NIfTI compresso
    sitk.WriteImage(img_ras, str(output_nifti_path), useCompression=True)
    
    print(f"Volume salvato con successo in: {output_nifti_path}")
    return img_ras


dataset_folders = sorted([f for f in BASE_DIR.glob("dataset*") if f.is_dir()])

print(f"Verifica percorso BASE_DIR: {BASE_DIR.exists()}")
all_folders = list(BASE_DIR.iterdir())
print(f"Cartelle totali trovate in BASE_DIR: {len(all_folders)}")

dataset_folders = sorted([f for f in BASE_DIR.glob("dataset*") if f.is_dir()])
print(f"Cartelle che corrispondono a 'dataset*': {len(dataset_folders)}")

if not dataset_folders:
    print("ERRORE: Nessuna cartella trovata. Controlla il nome del pattern 'dataset*' o il percorso BASE_DIR.")

for folder in dataset_folders:
    mhd_files = list(folder.glob("*.mhd"))
    
    if not mhd_files:
        print(f"Attenzione: nessun file .mhd trovato in {folder}")
        continue
        
    mhd_path = mhd_files[0]
    patient_id = folder.name 
    
    # Percorso di output standardizzato
    output_filename = OUTPUT_DIR / f"{patient_id}_volume.nii.gz"
    
    # Conversione
    convert_volume_to_ras_nifti(mhd_path, output_filename)

<h2> Creazione Json label <h2>

In [ ]:
import json
import SimpleITK as sitk

for folder in dataset_folders:
    patient_id = folder.name
    nifti_path = OUTPUT_DIR / f"{patient_id}_volume.nii.gz"
    
    if not nifti_path.exists():
        print(f"⚠️ Salto {patient_id}: volume non trovato.")
        continue
        
    img_ras = sitk.ReadImage(str(nifti_path))
    
    # --- NUOVA STRUTTURA METADATI ---
    patient_labels = {
        "id_immagine": patient_id,
        "spacing": list(img_ras.GetSpacing()),
        "orientation": list(img_ras.GetDirection()),
        "origin": list(img_ras.GetOrigin()),
        "ostia": {}, # Qui metteremo i punti RAS/IJK
        "comment": "",
        "other": {}
    }
    
    vessel_mapping = {0: "R", 1: "L"}
    
    for v_idx, side in vessel_mapping.items():
        vessel_dir = folder / f"vessel{v_idx}"
        target_files = ["pointS.txt", "reference.txt"]
        
        for filename in target_files:
            txt_path = vessel_dir / filename
            if txt_path.exists():
                with open(txt_path, 'r') as f:
                    lines = f.readlines()
                    if not lines: continue
                    
                    first_line = lines[0].strip()
                    if not first_line: continue
                    
                    try:
                        # 1. Coordinate originali (LPS)
                        lps_coords = [float(c) for c in first_line.split()[:3]]
                        
                        # 2. Conversione in RAS
                        ras_coords = [-lps_coords[0], -lps_coords[1], lps_coords[2]]
                        
                        # 3. Trasformazione in indici Voxel (IJK)
                        voxel_idx = img_ras.TransformPhysicalPointToIndex(lps_coords)
                        
                        # Salvataggio nel formato richiesto
                        patient_labels["ostia"][side] = {
                            "coordinate_ras": ras_coords,
                            "coordinate_ijk": list(voxel_idx),
                            "world_mm_original_lps": lps_coords, 
                            "source_file": filename
                        }
                        break 
                    except Exception as e:
                        print(f"⚠️ Errore in {txt_path}: {e}")
        
    # Salvataggio del JSON
    if patient_labels["ostia"]:
        json_filename = OUTPUT_DIR / f"{patient_id}_labels.json"
        with open(json_filename, 'w', encoding='utf-8') as jf:
            json.dump(patient_labels, jf, indent=4)
        print(f"✅ JSON completo salvato per {patient_id}")

<h3> Creazione mrk.json per check 3D slicer <h3>

In [ ]:
import json
from pathlib import Path


def create_slicer_markup(patient_id, data):
    
    # Template base per Slicer Markups
    markup_template = {
        "@schema": "https://raw.githubusercontent.com/slicer/slicer/master/Modules/Loadable/Markups/Resources/Schema/markups-schema-v1.0.3.json#",
        "markups": [
            {
                "type": "Fiducial",
                "coordinateSystem": "RAS", 
                "coordinateUnits": "mm",
                "controlPoints": [],
                "display": {
                    "visibility": True,
                    "opacity": 1.0,
                    "glyphScale": 2.5,
                    "glyphType": "Sphere3D",
                    "labelVisibility": True
                }
            }
        ]
    }

    
    ostia_info = data.get("ostia", {})
    
    for side, info in ostia_info.items():
    
        pos = info.get("coordinate_ras")
        
        if pos is None:
            print(f"⚠️ Salto {side} per {patient_id}: coordinata_ras non trovata.")
            continue

        # Colore: Rosso per Right (R), Blu per Left (L)
        
        point_color = [1.0, 0.0, 0.0] if side == "R" else [0.0, 0.0, 1.0]
        
        point = {
            "id": f"{patient_id}_{side}",
            "label": f"{side}",
            "description": f"Verifica RAS - Sorgente: {info.get('source_file', 'N/A')}",
            "position": pos,
            "orientation": [-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0],
            "selected": True,
            "locked": False,
            "visibility": True,
            "positionStatus": "defined",
            "color": point_color
        }
        markup_template["markups"][0]["controlPoints"].append(point)
    
    return markup_template

# --- ESECUZIONE ---

json_files = list(OUTPUT_DIR.glob("*_labels.json"))

if not json_files:
    print(f"❌ Errore: Nessun file JSON trovato in {OUTPUT_DIR}. Verifica il blocco precedente!")
else:
    print(f"🔍 Trovati {len(json_files)} file JSON. Generazione markup per Slicer...")

    for json_path in json_files:
       
        patient_id = json_path.name.replace("_labels.json", "")
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Genera la struttura .mrk.json
            slicer_markup = create_slicer_markup(patient_id, data)
            
            # Salvataggio
            output_path = OUTPUT_DIR / f"{patient_id}_check.mrk.json"
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(slicer_markup, f, indent=4)
            
            print(f"📍 Creato: {output_path.name}")
            
        except Exception as e:
            print(f"❌ Errore durante il processamento di {patient_id}: {e}")

    print("\n✅ Operazione completata.")
    